# License Plate Detection — SSD300 VGG16 (Google Colab)
**CMPS 261 — Machine Learning Project**

Run on Colab **T4/A100 GPU**. Make sure `license_plate_data.zip` is in your Google Drive root.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SSD300 VGG16 — License Plate Detection  (CMPS 261)
# ─────────────────────────────────────────────────────────────────────────────

# 1. MOUNT DRIVE & EXTRACT DATA
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
zip_path = '/content/drive/MyDrive/license_plate_data.zip'
if not os.path.exists('/content/data/yolo'):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print('Extracted.')
else:
    print('Already extracted.')

TRAIN_IMG = '/content/data/yolo/images/train'
VAL_IMG   = '/content/data/yolo/images/val'
TEST_IMG  = '/content/data/yolo/images/test'
TRAIN_LBL = '/content/data/yolo/labels/train'
VAL_LBL   = '/content/data/yolo/labels/val'
TEST_LBL  = '/content/data/yolo/labels/test'
print(f'Train: {len(os.listdir(TRAIN_IMG))} | Val: {len(os.listdir(VAL_IMG))} | Test: {len(os.listdir(TEST_IMG))}')

# 2. GPU CHECK
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}' + (f'  ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else ''))

# 3. DATASET CLASS (YOLO txt → SSD format)
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF
import random

class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, augment=False):
        self.augment = augment
        self.samples = []
        for lbl_file in sorted(os.listdir(lbl_dir)):
            if not lbl_file.endswith('.txt'):
                continue
            stem = os.path.splitext(lbl_file)[0]
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = os.path.join(img_dir, stem + ext)
                if os.path.exists(img_path):
                    self.samples.append((img_path, os.path.join(lbl_dir, lbl_file)))
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        # SSD300 expects 300x300 input
        img = img.resize((300, 300))
        W_orig = Image.open(img_path).size[0]
        H_orig = Image.open(img_path).size[1]

        boxes = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                _, cx, cy, w, h = map(float, parts[:5])
                # Convert normalized YOLO → absolute coords in 300x300 space
                xmin = max(0.0, (cx - w / 2) * 300)
                ymin = max(0.0, (cy - h / 2) * 300)
                xmax = min(300.0, (cx + w / 2) * 300)
                ymax = min(300.0, (cy + h / 2) * 300)
                if xmax > xmin and ymax > ymin:
                    boxes.append([xmin, ymin, xmax, ymax])

        if not boxes:
            boxes = [[0.0, 0.0, 1.0, 1.0]]

        if self.augment:
            if random.random() > 0.5:
                img   = TF.hflip(img)
                boxes = [[300 - b[2], b[1], 300 - b[0], b[3]] for b in boxes]
            img = TF.adjust_brightness(img, 1 + random.uniform(-0.2, 0.2))
            img = TF.adjust_contrast(img,   1 + random.uniform(-0.2, 0.2))
            img = TF.adjust_saturation(img, 1 + random.uniform(-0.2, 0.2))

        boxes  = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones(len(boxes), dtype=torch.int64)
        target = {
            'boxes'   : boxes,
            'labels'  : labels,
        }
        return TF.to_tensor(img), target

def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = LicensePlateDataset(TRAIN_IMG, TRAIN_LBL, augment=True)
val_ds   = LicensePlateDataset(VAL_IMG,   VAL_LBL,   augment=False)
test_ds  = LicensePlateDataset(TEST_IMG,  TEST_LBL,  augment=False)
print(f'Loaded — Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)

# 4. BUILD MODEL
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead
from functools import partial

weights = SSD300_VGG16_Weights.DEFAULT
model   = ssd300_vgg16(weights=weights)

# Replace classification head for 2 classes (background + licence)
num_anchors = model.anchor_generator.num_anchors_per_location()
in_channels = [512, 1024, 512, 256, 256, 256]
model.head.classification_head = SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=2,
)
model.to(DEVICE)

params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.002, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
print('Model ready.')

# 5. TRAIN WITH EARLY STOPPING
import time

MODEL_PATH = '/content/ssd_best.pth'
best_val   = float('inf')
no_improve = 0
PATIENCE   = 8
NUM_EPOCHS = 50

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    model.train()
    total_train = 0
    for images, targets in train_loader:
        images  = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train += loss.item()
    train_loss = total_train / len(train_loader)

    model.train()
    total_val = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            total_val += sum(loss_dict.values()).item()
    val_loss = total_val / len(val_loader)

    scheduler.step(val_loss)

    flag = ''
    if val_loss < best_val:
        best_val   = val_loss
        no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
        flag = ' <- best'
    else:
        no_improve += 1

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | {time.time()-t0:.0f}s{flag}')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

print('\nTraining complete!')

# 6. FIND BEST THRESHOLD & EVALUATE
import numpy as np

def compute_iou(a, b):
    xA, yA = max(a[0],b[0]), max(a[1],b[1])
    xB, yB = min(a[2],b[2]), min(a[3],b[3])
    inter  = max(0, xB-xA) * max(0, yB-yA)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-6)

def evaluate(loader, threshold):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    tp, fp, fn = 0, 0, 0
    iou_scores = []
    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(DEVICE) for img in images]
            preds  = model(images)
            for pred, target in zip(preds, targets):
                gt_boxes   = target['boxes'].numpy()
                pred_boxes = pred['boxes'][pred['scores'] >= threshold].cpu().numpy()
                matched = set()
                for pb in pred_boxes:
                    best_iou, best_j = 0, -1
                    for j, gb in enumerate(gt_boxes):
                        iou = compute_iou(pb, gb)
                        if iou > best_iou: best_iou, best_j = iou, j
                    if best_iou >= 0.5 and best_j not in matched:
                        tp += 1; matched.add(best_j); iou_scores.append(best_iou)
                    else:
                        fp += 1
                fn += len(gt_boxes) - len(matched)
    precision = tp / (tp + fp + 1e-6)
    recall    = tp / (tp + fn + 1e-6)
    f1        = 2 * precision * recall / (precision + recall + 1e-6)
    mean_iou  = float(np.mean(iou_scores)) if iou_scores else 0.0
    return precision, recall, f1, mean_iou

print('\nFinding best confidence threshold...')
best_f1, best_thresh = 0, 0.5
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8]:
    p, r, f, _ = evaluate(val_loader, thresh)
    print(f'  thresh={thresh} | P={p:.3f} R={r:.3f} F1={f:.3f}')
    if f > best_f1:
        best_f1, best_thresh = f, thresh

print(f'\nBest threshold: {best_thresh}')
precision, recall, f1, mean_iou = evaluate(test_loader, best_thresh)

print(f'\n=== Test Results (threshold={best_thresh}) ===')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1        : {f1:.4f}')
print(f'Mean IoU  : {mean_iou:.4f}')

# 7. SAVE METRICS & DOWNLOAD
import json
from google.colab import files

metrics = {
    'model'    : 'SSD300 VGG16',
    'epochs'   : epoch,
    'threshold': best_thresh,
    'precision': round(precision, 4),
    'recall'   : round(recall,    4),
    'f1'       : round(f1,        4),
    'mean_iou' : round(mean_iou,  4),
}
with open('/content/ssd_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

files.download('/content/ssd_best.pth')
files.download('/content/ssd_metrics.json')